<a href="https://colab.research.google.com/github/fdhliakbar/IR-Lab/blob/main/P06_Evaluasi_IR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum ke 6 - Evaluasi IR



Dalam Information Retrieval (IR), evaluasi performa adalah langkah penting untuk mengukur seberapa baik sistem pencarian dalam menyediakan dokumen yang relevan terhadap suatu kueri. Tiga metrik evaluasi yang sering digunakan adalah Mean Average Precision (MAP), Precision @K, dan R-Precision.

In [6]:
# import library standard
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [7]:
!pip install scikit-learn

In [8]:
# import library sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB

# import library sklearn (Evaluasi tak berperingkat)
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

# import library sklearn (Evaluasi berperingkat)
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score

In [9]:
# import library untuk stemming
!pip install Sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

In [10]:
# read dataset
data = pd.read_excel('/content/dataKumparan1.xlsx')
data.head()

,Topic,Title,Content
0,Politik,"Pelanggaran Pemilu, Tiga Caleg di Sulteng Dipr...","Komisioner Bawaslu Sigi, Sulawesi Tengah, Agus..."
1,Politik,"Pemilu Susulan di Kota Jayapura, Suara Jokowi ...",Walaupun dua dari lima distrik melakukan pemil...
2,Politik,"Tsamara Amany Dipinang, Pengurus PSI Daerah Me...","Tsamara Amany, politisi Partai Solidaritas Ind..."
3,Politik,Ada 47 TPS di Sulawesi Utara Berpotensi Pemili...,Badan Pengawas Pemilu (Bawaslu) Provinsi Sulaw...
4,Politik,Ketua KPPS di Sleman Ditemukan Tewas Gantung D...,"Tugiman, Ketua Kelompok Penyelenggara Pemungut..."


In [11]:
# ukuran dataset
print('Ukuran dataset: ', data.shape)

Ukuran dataset:  (60, 3)


In [12]:
# pembagian data training & testing
x_train, x_test, y_train, y_test = train_test_split(data['Content'], data['Topic'], train_size = 0.5, test_size = 0.16)

In [13]:
train_data = pd.DataFrame({'Content': x_train, 'Topic': y_train})
test_data = pd.DataFrame({'Content': x_test, 'Topic': y_test})

In [14]:
df1 = pd.DataFrame(train_data)
print(df1)

                                              Content      Topic
8   Pertamuan para kiai sepuh se-Jawa Timur yang d...    Politik
42  Dinas Kebudayaan dan Pariwisata (Disbudpar) Ko...     Travel
55  Maskapai bersimbol singa merah, Lion Air kerap...     Travel
59  Di balik kemegahan Pegunungan Tianzhu China, a...     Travel
50  Untuk pertama kalinya dalam 300 tahun, Vatikan...     Travel
27  Pesta demokrasi terbesar di Indonesia resmi di...  Teknologi
46  Kebakaran terjadi di terminal keberangkatan do...     Travel
4   Tugiman, Ketua Kelompok Penyelenggara Pemungut...    Politik
14  KPU memberikan klarifikasi mengenai banyaknya ...    Politik
15  Mantan Ketua Mahkamah Konstitusi (MK), Mahfud ...    Politik
38  Kamu mungkin pernah merasa kesulitan untuk ber...  Teknologi
39  Perusahaan e-commerce marketplace Tokopedia ke...  Teknologi
41  Berlibur menjadi kegiatan yang paling dinanti ...     Travel
44  Hari Guru atau Minggu Apreasi Guru di Amerika ...     Travel
20  Facebook sedang menge

In [15]:
df2 = pd.DataFrame(test_data)
print(df2)

                                              Content      Topic
24  Stasiun Kereta Api Stockholm, Swedia, merupaka...  Teknologi
12  Jalan Kapas Madya Barat 9 Nomor 38, Tambak Sar...    Politik
25  Apple ternyata tidak main-main untuk terjun ke...  Teknologi
43  Anda suka dessert? Di berbagai media sosial, p...     Travel
21  Facebook hari Rabu mengaku telah secara tidak ...  Teknologi
37  Belanda merupakan negara yang secara geografis...  Teknologi
16  Pemungutan suara Pemilu 2019 telah usai. Tapi ...    Politik
7   Badan Pengawas Pemilu (Bawaslu) Kota Banjarmas...    Politik
2   Tsamara Amany, politisi Partai Solidaritas Ind...    Politik
51  Tahun 2018 lalu, keluarga besar Kim Kardashian...     Travel


In [16]:
print('ukuran data train: ', train_data.shape)
print('ukuran data test: ', test_data.shape)
n_train = train_data.shape[0]
n_test = test_data.shape[0]

ukuran data train:  (30, 2)
ukuran data test:  (10, 2)


In [17]:
sparse_data = pd.concat([train_data, test_data], ignore_index=True)
df3 = pd.DataFrame(sparse_data)
print(df3)

                                              Content      Topic
0   Pertamuan para kiai sepuh se-Jawa Timur yang d...    Politik
1   Dinas Kebudayaan dan Pariwisata (Disbudpar) Ko...     Travel
2   Maskapai bersimbol singa merah, Lion Air kerap...     Travel
3   Di balik kemegahan Pegunungan Tianzhu China, a...     Travel
4   Untuk pertama kalinya dalam 300 tahun, Vatikan...     Travel
5   Pesta demokrasi terbesar di Indonesia resmi di...  Teknologi
6   Kebakaran terjadi di terminal keberangkatan do...     Travel
7   Tugiman, Ketua Kelompok Penyelenggara Pemungut...    Politik
8   KPU memberikan klarifikasi mengenai banyaknya ...    Politik
9   Mantan Ketua Mahkamah Konstitusi (MK), Mahfud ...    Politik
10  Kamu mungkin pernah merasa kesulitan untuk ber...  Teknologi
11  Perusahaan e-commerce marketplace Tokopedia ke...  Teknologi
12  Berlibur menjadi kegiatan yang paling dinanti ...     Travel
13  Hari Guru atau Minggu Apreasi Guru di Amerika ...     Travel
14  Facebook sedang menge

In [18]:
# ukuran sparse data
print('ukuran data test: ', sparse_data.shape)
n_document = sparse_data.shape[0]

ukuran data test:  (40, 2)


# **Preprocessing dengan Stemming dan Stopword**

In [19]:
# create stemmer
StemmerFactory = StemmerFactory()
stemmer = StemmerFactory.create_stemmer()

In [20]:
# stem process
for row in range(n_document):
  sparse_data.loc[row, 'Content'] = stemmer.stem(sparse_data.loc[row, 'Content'])

In [21]:
df4 = pd.DataFrame(sparse_data)
print(df4)

                                              Content      Topic
0   tamu para kiai sepuh se-jawa timur yang gelar ...    Politik
1   dinas budaya dan pariwisata disbudpar kota mal...     Travel
2   maskapai simbol singa merah lion air kerap kal...     Travel
3   di balik megah gunung tianzhu china ada buah d...     Travel
4   untuk pertama kali dalam 300 tahun vatikan aka...     Travel
5   pesta demokrasi besar di indonesia resmi gelar...  Teknologi
6   bakar jadi di terminal berangkat domestik band...     Travel
7   tugiman ketua kelompok selenggara mungut suara...    Politik
8   kpu beri klarifikasi kena banyak temu salah in...    Politik
9   mantan ketua mahkamah konstitusi mk mahfud md ...    Politik
10  kamu mungkin pernah rasa sulit untuk komunikas...  Teknologi
11  usaha e-commerce marketplace tokopedia kembali...  Teknologi
12  libur jadi giat yang paling nanti oleh banyak ...     Travel
13  hari guru atau minggu apreasi guru di amerika ...     Travel
14  facebook sedang kemba

# **Perhitungan Bobot**

In [22]:
vectorizer = CountVectorizer()
tf = vectorizer.fit_transform(sparse_data['Content'])
print(['Jumlah dokumen: ', tf.shape[0]])
print(['Jumlah term: ', tf.shape[1]])

['Jumlah dokumen: ', 40]
['Jumlah term: ', 2292]


In [23]:
print('Daftar Term:')
vectorizer.get_feature_names_out()

Daftar Term:


array(['00', '000', '0004', ..., 'zamih', 'ziarah', 'zoetry'],
      dtype=object)

In [24]:
print('Daftar Stopword:')
vectorizer.get_stop_words()

Daftar Stopword:


In [25]:
print('Matriks Tf:')
tf_matrix = pd.DataFrame(tf.toarray(), columns=vectorizer.get_feature_names_out())
tf_matrix

Matriks Tf:


,00,000,0004,01,02,04,08,09,10,100,...,yerusalem,yesus,yogyakarta,yunani,yusuf,zahid,zainuddin,zamih,ziarah,zoetry
0,0,0,0,1,2,0,0,0,0,0,...,0,0,0,0,1,0,1,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,1,5,0,0,0,0,0,0,5,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,1,0,0,0,0,0,0,2,0,0,...,0,0,1,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [26]:
print('Matriks Tf (khusus data train):')
tf_train = tf_matrix[:n_train]
tf_train.shape

Matriks Tf (khusus data train):


(30, 2292)

In [27]:
transformer = TfidfTransformer(sublinear_tf=True)

# Penyesuaian df agar query (data test) tidak dihitung pada perhitungan df
n = n_train
df = tf_train.astype(bool).sum(axis=0)
idf = np.log(n/df)
transformer.idf_ = idf

weight = transformer.fit_transform(tf)
print('Jumlah dokumen:', weight.shape[0])
print('Jumlah term:', weight.shape[1])

Jumlah dokumen: 40
Jumlah term: 2292


In [28]:
weight_matrix = pd.DataFrame(weight.toarray(), columns=vectorizer.get_feature_names_out())
weight_matrix

,00,000,0004,01,02,04,08,09,10,100,...,yerusalem,yesus,yogyakarta,yunani,yusuf,zahid,zainuddin,zamih,ziarah,zoetry
0,0.000000,0.000000,0.000000,0.057421,0.108128,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.063862,0.000000,0.063862,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.052582,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.066305,0.173018,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.173018,0.000000
5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,0.000000,0.000000,0.000000,0.000000,0.000000,0.092334,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,0.079958,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.147086,0.000000,0.000000,...,0.000000,0.000000,0.096615,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,0.000000,0.000000,0.074742,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [29]:
# pembagian matriks bobot
weight_train = weight_matrix[:n_train]
weight_test = weight_matrix[n_train:]

# **Perhitungan Cosine Similarity**

In [30]:
# perhitungan Cosine Similarity
cosim = cosine_similarity(weight_train, weight_test)
print('Ukuran matriks cosine similarity:', cosim.shape)

Ukuran matriks cosine similarity: (30, 10)


In [31]:
name = []
for i in range(n_test):
  name.append('Dokumen ' + str(i))

In [32]:
# baris = dokumen train, kolom = dokumen test
print('Matriks Cosine Similarity:')
cosim_matrix = pd.DataFrame(cosim, columns=name)
cosim_matrix.shape

Matriks Cosine Similarity:


(30, 10)

In [33]:
train_data

,Content,Topic
8,Pertamuan para kiai sepuh se-Jawa Timur yang d...,Politik
42,Dinas Kebudayaan dan Pariwisata (Disbudpar) Ko...,Travel
55,"Maskapai bersimbol singa merah, Lion Air kerap...",Travel
59,"Di balik kemegahan Pegunungan Tianzhu China, a...",Travel
50,"Untuk pertama kalinya dalam 300 tahun, Vatikan...",Travel
27,Pesta demokrasi terbesar di Indonesia resmi di...,Teknologi
46,Kebakaran terjadi di terminal keberangkatan do...,Travel
4,"Tugiman, Ketua Kelompok Penyelenggara Pemungut...",Politik
14,KPU memberikan klarifikasi mengenai banyaknya ...,Politik
15,"Mantan Ketua Mahkamah Konstitusi (MK), Mahfud ...",Politik


In [34]:
# cosim matrix + label
cosim_matrix['Label Train'] = train_data['Topic'].values
label_row = dict(zip(name, test_data['Topic'].values))
label_cosim = pd.concat([cosim_matrix, pd.DataFrame([label_row], index=['Label Test'])], ignore_index=False)
label_cosim.rename({label_cosim.index[-1]:'Label Test'}, inplace=True)


label_test = pd.DataFrame(label_cosim.iloc[-1])
label_test = label_test.T
label_test

,Dokumen 0,Dokumen 1,Dokumen 2,Dokumen 3,Dokumen 4,Dokumen 5,Dokumen 6,Dokumen 7,Dokumen 8,Dokumen 9,Label Train
Label Test,Teknologi,Politik,Teknologi,Travel,Teknologi,Teknologi,Politik,Politik,Politik,Travel,NaN


In [35]:
label_cosim

,Dokumen 0,Dokumen 1,Dokumen 2,Dokumen 3,Dokumen 4,Dokumen 5,Dokumen 6,Dokumen 7,Dokumen 8,Dokumen 9,Label Train
0,0.071206,0.078532,0.053255,0.089739,0.055938,0.061349,0.092424,0.133678,0.0609,0.062571,Politik
1,0.082472,0.077184,0.075957,0.109838,0.078209,0.072941,0.077909,0.126338,0.065504,0.074546,Travel
2,0.117474,0.076568,0.133146,0.075323,0.08822,0.123309,0.057084,0.116863,0.0498,0.081969,Travel
3,0.112093,0.069035,0.087157,0.090631,0.070109,0.13689,0.061713,0.109836,0.075376,0.08448,Travel
4,0.099933,0.05779,0.084408,0.088685,0.093432,0.117284,0.079379,0.071405,0.075529,0.127017,Travel
5,0.065545,0.081949,0.067513,0.064264,0.081467,0.073194,0.090992,0.05546,0.08729,0.127082,Teknologi
6,0.072112,0.068069,0.066342,0.058847,0.063466,0.059969,0.088415,0.06877,0.048143,0.095712,Travel
7,0.059358,0.139973,0.019035,0.03996,0.061055,0.068965,0.115426,0.115365,0.087482,0.080366,Politik
8,0.078828,0.110857,0.065303,0.053482,0.076368,0.07839,0.176655,0.196966,0.101894,0.084589,Politik
9,0.076756,0.142274,0.082138,0.065369,0.103632,0.079963,0.165932,0.238144,0.123508,0.084563,Politik


In [36]:
cosim_matrix

,Dokumen 0,Dokumen 1,Dokumen 2,Dokumen 3,Dokumen 4,Dokumen 5,Dokumen 6,Dokumen 7,Dokumen 8,Dokumen 9,Label Train
0,0.071206,0.078532,0.053255,0.089739,0.055938,0.061349,0.092424,0.133678,0.060900,0.062571,Politik
1,0.082472,0.077184,0.075957,0.109838,0.078209,0.072941,0.077909,0.126338,0.065504,0.074546,Travel
2,0.117474,0.076568,0.133146,0.075323,0.088220,0.123309,0.057084,0.116863,0.049800,0.081969,Travel
3,0.112093,0.069035,0.087157,0.090631,0.070109,0.136890,0.061713,0.109836,0.075376,0.084480,Travel
4,0.099933,0.057790,0.084408,0.088685,0.093432,0.117284,0.079379,0.071405,0.075529,0.127017,Travel
5,0.065545,0.081949,0.067513,0.064264,0.081467,0.073194,0.090992,0.055460,0.087290,0.127082,Teknologi
6,0.072112,0.068069,0.066342,0.058847,0.063466,0.059969,0.088415,0.068770,0.048143,0.095712,Travel
7,0.059358,0.139973,0.019035,0.039960,0.061055,0.068965,0.115426,0.115365,0.087482,0.080366,Politik
8,0.078828,0.110857,0.065303,0.053482,0.076368,0.078390,0.176655,0.196966,0.101894,0.084589,Politik
9,0.076756,0.142274,0.082138,0.065369,0.103632,0.079963,0.165932,0.238144,0.123508,0.084563,Politik


In [37]:
# prompt: buatkan tampilan cosim_matrix secara descending

# Sort the cosim_matrix by all columns in descending order
cosim_matrix_sorted = cosim_matrix.sort_values(by=cosim_matrix.columns.tolist(), ascending=False)
cosim_matrix_sorted

,Dokumen 0,Dokumen 1,Dokumen 2,Dokumen 3,Dokumen 4,Dokumen 5,Dokumen 6,Dokumen 7,Dokumen 8,Dokumen 9,Label Train
10,0.147126,0.078234,0.114101,0.103716,0.092271,0.114474,0.084880,0.091088,0.079517,0.087309,Teknologi
12,0.121685,0.060364,0.123749,0.107034,0.088584,0.112190,0.074899,0.091087,0.087154,0.133970,Travel
19,0.119485,0.074270,0.096838,0.078928,0.090184,0.155844,0.094804,0.080791,0.089006,0.135721,Travel
21,0.117804,0.063547,0.116895,0.090899,0.115715,0.091373,0.095626,0.106569,0.085963,0.126301,Travel
2,0.117474,0.076568,0.133146,0.075323,0.088220,0.123309,0.057084,0.116863,0.049800,0.081969,Travel
17,0.115745,0.060319,0.124727,0.110161,0.075356,0.107571,0.076721,0.085093,0.077351,0.076479,Teknologi
3,0.112093,0.069035,0.087157,0.090631,0.070109,0.136890,0.061713,0.109836,0.075376,0.084480,Travel
18,0.101117,0.037853,0.167685,0.079511,0.106258,0.121971,0.062448,0.080042,0.088605,0.085849,Teknologi
4,0.099933,0.057790,0.084408,0.088685,0.093432,0.117284,0.079379,0.071405,0.075529,0.127017,Travel
22,0.092796,0.082195,0.087146,0.060080,0.059972,0.098353,0.073342,0.096523,0.084810,0.095680,Travel


In [38]:
import os

directory = '/content/drive/MyDrive/STBI'
if not os.path.exists(directory):
    os.makedirs(directory)

cosim_matrix.to_csv(os.path.join(directory, 'cosim_matrix.csv'), index=False)
cosim_matrix_sorted.to_csv(os.path.join(directory, 'cosim_matrix_sorted.csv'), index=False)

In [39]:
def average_precision(cosim_matrix, n_retrieve):
  average_precision = []

  # loop untuk mengambil setiap query di cosim matrix
  for column in cosim_matrix.iloc[:, :-1]:
    # sort and get top n
    sorted_cosim = cosim_matrix.sort_values(column, ascending=False)
    top_n = sorted_cosim.iloc[:n_retrieve]
    # print(top_n)

    relevant = (np.array(top_n['Label Train']) == np.array(label_test[column]))
    # print(relevant)

    # list of precision
    precision = []
    peringkat = 0
    counter_relevant = 0

    for r in relevant:
      peringkat +=1
      if r == True:
        counter_relevant += 1
        precision.append(counter_relevant/peringkat)

    average_precision.append(np.mean(precision))

    # print('Average Precision: ', np.mean(precision))
    # print(average_precision)

  return average_precision

def mean_average_precision(cosim_matrix, n_retrieve):
  ap = average_precision(cosim_matrix, n_retrieve)
  map = np.mean(ap)
  return map

def precision_at_k(cosim_matrix, k_retrieve):
  precision_at_k = []

  # loop utk mengambil setiap query di cosim matrix
  for column in cosim_matrix.iloc[:, :-1]:
    # sort and get top n
    sorted_cosim = cosim_matrix.sort_values(column, ascending=False)
    top_n = sorted_cosim.iloc[:k_retrieve]
    # print(top_n)

    # list of relevance
    relevant = (np.array(top_n['Label Train']) == np.array(label_test[column]))
    # print(relevant)

    # list of precision
    precision = np.sum(relevant)/len(relevant)
    precision_at_k.append(precision)
    # print(precision_at_k)

  return precision_at_k

def r_precision(cosim_matrix, n_retrieve):
    r_precision = []

    # loop utk mengambil setiap query di cosim matrix
    for column in cosim_matrix.iloc[:, :-1]:
      # sort and get top n
      sorted_cosim = cosim_matrix.sort_values(column, ascending=False)
      top_n = sorted_cosim.iloc[:n_retrieve]
      # print(top_n)

      # list of relevance
      relevant = (np.array(top_n['Label Train']) == np.array(label_test[column]))
      # print(relevant

      # list of precision
      precision = np.sum(relevant)/len(relevant)
      r_precision.append(precision)
    return r_precision

In [40]:
print('\nHasil Evaluasi MAP: ', mean_average_precision(cosim_matrix, 10))
print('Hasil Evaluasi Precision@k: ', precision_at_k(cosim_matrix, 10))
print('Hasil Evaluasi R-Precision: ', r_precision(cosim_matrix, 10))


Hasil Evaluasi MAP:  0.7067681878306878
Hasil Evaluasi Precision@k:  [np.float64(0.3), np.float64(0.5), np.float64(0.7), np.float64(0.6), np.float64(0.5), np.float64(0.3), np.float64(0.5), np.float64(0.6), np.float64(0.4), np.float64(0.8)]
Hasil Evaluasi R-Precision:  [np.float64(0.3), np.float64(0.5), np.float64(0.7), np.float64(0.6), np.float64(0.5), np.float64(0.3), np.float64(0.5), np.float64(0.6), np.float64(0.4), np.float64(0.8)]


In [41]:
numbers = [
    0.164109, 0.054123, 0.138920,
    0.084240, 0.084468, 0.066134,
    0.063556
]

mean_value = sum(numbers) / 10
mean_value


0.065555

## POSTEST 6 - Evaluasi IR

Analisa dari dataset yang dimasukkan apakah sesuai
dengan perhitungan?  

Note: Jika ada perlu ditanyakan terkait teknis praktikum, jangan ragu bertanya. Silahkan bertanya di group atau pc dengan asisten `Fadhli` & `Aufa`

### Selamat Mengerjakan 😺